# Task 5/6: Impact of Budget Misestimation (Section 3.4)

This notebook analyses what happens when a planner designs the facility network under an **assumed disruption budget** Γ_assumed that may differ from the **true** budget Γ_true = 2.

The uncertainty set is defined by equation (11) of the paper:

$$\Xi(\Gamma) = \left\{ \varepsilon \in \{0,1\}^{J \times H} \;\Big|\; \sum_h \varepsilon_{jh} = 1\;\forall j,\; \sum_j \sum_h \frac{h}{H-1}\varepsilon_{jh} \le \Gamma \right\}$$

We solve the two-stage robust CFLP for each Γ_assumed ∈ {0, 1, 2, 3, 4} and then evaluate the resulting first-stage solution x_jr(Γ_assumed) under:
1. **No disruption** — the “peace-time” baseline (shows the cost of conservatism).
2. **Worst-case disruption** at the TRUE budget Γ_true = 2 — the adversarial test.
3. **Average-case disruption** sampled from Ξ(Γ_true) — a Monte Carlo estimate.

Two key metrics are defined:

- **Cost of underestimation** (Γ_assumed < Γ_true): profit of the correctly-specified design minus profit of the under-specified design, both evaluated under the true worst-case disruption. Underestimation leaves the network exposed to disruptions it was not hardened against.
- **Cost of overestimation** (Γ_assumed > Γ_true): profit of the correctly-specified design minus profit of the over-specified design, both evaluated under no disruption. Overestimation incurs unnecessarily high fixed costs, reducing peace-time profitability.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from rcflp import (
    instancemaker,
    solve_nominal,
    solve_CCG,
    evaluate_second_stage,
    worst_case_disruption,
    sample_disruptions,
    no_disruption_scenario,
)

## 1. Configuration

In [ ]:
GAMMA_TRUE    = 2          # true disruption budget faced in operation
GAMMA_ASSUMED = [0, 1, 2, 3, 4]  # planner’s assumed budgets at design time
Hn            = 2          # number of disruption levels (H = {0, 1})
In, Jn, Rn    = 10, 8, 2   # customers, candidate facilities, capacity levels
V_SCALE       = 0.75       # value_max = V_SCALE * max_distance
W             = 10.0       # congestion cost rate (uniform)
N_SAMPLES     = 50         # Monte Carlo sample size
SEED          = 42         # random seed for reproducibility
TOL           = 0.01       # C&CG optimality tolerance
TIME_LIMIT    = 600        # solver time limit (seconds)
DATA_PATH     = '../dataset.xlsx'

## 2. Solve robust problem for each assumed budget

For Γ_assumed = 0 the problem is the **nominal** (no robustness); we call `solve_nominal` directly.
For Γ_assumed > 0 we run the **Column-and-Constraint Generation (C&CG)** algorithm with the corresponding budget.
This produces five distinct first-stage facility-opening decisions x_jr(Γ_assumed).
Higher assumed budgets yield more conservative (expensive, dispersed) designs that are robust to more severe disruptions but sacrifice peace-time profit.

In [ ]:
inst  = instancemaker(In, Jn, Rn, V_SCALE, W, data_path=DATA_PATH)
nom   = solve_nominal(inst)
x_nom = nom['x_jr']
print(f"Nominal profit: {nom['profit']:,.1f}  (runtime: {nom['runtime']:.1f}s)")

solutions = {}
for G_assumed in GAMMA_ASSUMED:
    print(f"\n--- Solving for Gamma_assumed = {G_assumed} ---")
    if G_assumed == 0:
        solutions[G_assumed] = {
            'x_jr': x_nom,
            'profit_design': nom['profit'],
            'label': 'Nominal (G=0)',
            'runtime': nom['runtime'],
            'converged': True,
        }
    else:
        res = solve_CCG(
            inst, G_assumed, Hn,
            x_init=x_nom,
            tol=TOL,
            time_limit=TIME_LIMIT,
            verbose=True,
        )
        solutions[G_assumed] = {
            'x_jr': res['x_jr'],
            'profit_design': res['profit_LB'],
            'label': f'Robust (G={G_assumed})',
            'runtime': res['runtime'],
            'converged': res['converged'],
        }
    print(f"  Design profit: {solutions[G_assumed]['profit_design']:,.1f}  "
          f"(runtime: {solutions[G_assumed]['runtime']:.1f}s)")

print("\nAll designs solved.")

## 3. Pre-disruption performance

We evaluate each design under the **no-disruption scenario** (ε_j = 1 for all j, i.e., all facilities at full capacity).
This is the scenario that a planner optimising only for normal conditions would target.
More conservative designs (higher Γ_assumed) open more or larger facilities to ensure robust coverage, which increases fixed costs and thereby reduces peace-time profit — this is the **price of robustness** under normal operations.

In [ ]:
eps0 = no_disruption_scenario(inst, Hn)

rows_nodis = []
for G, sol in solutions.items():
    print(f"  Evaluating no-disruption for Gamma_assumed={G}...")
    res = evaluate_second_stage(inst, sol['x_jr'], eps0, Hn)
    rows_nodis.append({
        'Gamma_assumed':      G,
        'label':              sol['label'],
        'profit_nodisruption': res['profit'],
        'revenue':            res['breakdown']['revenue'],
        'fixed':              res['breakdown']['fixed_cost'],
        'congestion':         res['breakdown']['congestion'],
    })

df_nodis = pd.DataFrame(rows_nodis)
print()
print(df_nodis[['Gamma_assumed', 'label', 'profit_nodisruption',
                'revenue', 'fixed', 'congestion']].to_string(index=False))

## 4. Worst-case performance under TRUE budget Γ_true = 2

For each design we find the **adversarially worst disruption** ε* by solving the separation oracle with the TRUE budget Γ_true = 2, then evaluate second-stage profit under that scenario.

A design built for Γ_assumed < Γ_true has not seen disruptions of level 2 during optimisation, so the adversary can exploit its weaknesses more severely. A design built for Γ_assumed > Γ_true is over-hardened: it sustains fewer losses under the true worst case, but the protection comes at a cost already accounted for in fixed costs.

The **cost of misestimation** relative to the correctly-specified design (G=2) is computed.

In [ ]:
rows_wc = []
for G, sol in solutions.items():
    print(f"  Finding worst-case disruption for Gamma_assumed={G} (Gamma_true={GAMMA_TRUE})...")
    eps_wc, rc_wc = worst_case_disruption(inst, sol['x_jr'], GAMMA_TRUE, Hn)
    res = evaluate_second_stage(inst, sol['x_jr'], eps_wc, Hn)
    rows_wc.append({
        'Gamma_assumed':   G,
        'label':           sol['label'],
        'profit_worstcase': res['profit'],
        'revenue_wc':      res['breakdown']['revenue'],
        'congestion_wc':   res['breakdown']['congestion'],
    })
    print(f"    Worst-case profit: {res['profit']:,.1f}")

df_wc = pd.DataFrame(rows_wc)

# Cost of misestimation vs correctly-specified design (Gamma_assumed == GAMMA_TRUE)
correct_wc = df_wc.loc[df_wc['Gamma_assumed'] == GAMMA_TRUE, 'profit_worstcase'].values[0]
df_wc['cost_of_misestimation'] = correct_wc - df_wc['profit_worstcase']

print()
print(df_wc[['Gamma_assumed', 'label', 'profit_worstcase', 'cost_of_misestimation']].to_string(index=False))

## 5. Average-case performance under TRUE budget Γ_true = 2

We draw N_SAMPLES = 50 random disruption scenarios from Ξ(Γ_true) and compute each design’s **average profit** across those scenarios.
This gives a distributional view: beyond worst-case, how does each design perform on a typical disruption?

We also report the standard deviation, minimum, and maximum profit, giving a sense of each design’s resilience variability.

In [ ]:
print(f"Sampling {N_SAMPLES} disruption scenarios from Xi(Gamma_true={GAMMA_TRUE})...")
scenarios = sample_disruptions(inst, GAMMA_TRUE, Hn, N_SAMPLES, SEED)
print(f"Generated {len(scenarios)} scenarios.")

rows_avg = []
for G, sol in solutions.items():
    print(f"  Average-case evaluation for Gamma_assumed={G} ({N_SAMPLES} samples)...")
    profits = []
    for k, eps in enumerate(scenarios):
        res = evaluate_second_stage(inst, sol['x_jr'], eps, Hn)
        profits.append(res['profit'])
        if (k + 1) % 10 == 0:
            print(f"    {k+1}/{N_SAMPLES} scenarios done")
    rows_avg.append({
        'Gamma_assumed': G,
        'label':         sol['label'],
        'profit_avg':    np.mean(profits),
        'profit_std':    np.std(profits),
        'profit_min':    np.min(profits),
        'profit_max':    np.max(profits),
    })
    print(f"    Mean profit: {np.mean(profits):,.1f}  (std: {np.std(profits):,.1f})")

df_avg = pd.DataFrame(rows_avg)

correct_avg = df_avg.loc[df_avg['Gamma_assumed'] == GAMMA_TRUE, 'profit_avg'].values[0]
df_avg['cost_of_misestimation_avg'] = correct_avg - df_avg['profit_avg']

print()
print(df_avg[['Gamma_assumed', 'label', 'profit_avg', 'profit_std',
              'cost_of_misestimation_avg']].to_string(index=False))

## 6. Summary table

All three evaluation perspectives are merged into a single overview table.

In [ ]:
df_summary = (
    df_nodis[['Gamma_assumed', 'label', 'profit_nodisruption']]
    .merge(df_wc[['Gamma_assumed', 'profit_worstcase', 'cost_of_misestimation']],
           on='Gamma_assumed')
    .merge(df_avg[['Gamma_assumed', 'profit_avg', 'cost_of_misestimation_avg']],
           on='Gamma_assumed')
)

# Rename for display
df_summary = df_summary.rename(columns={
    'Gamma_assumed':          'Gamma_assumed',
    'label':                  'Design',
    'profit_nodisruption':    'Profit (no disrupt.)',
    'profit_worstcase':       'Profit (worst-case)',
    'cost_of_misestimation':  'Cost misest. (WC)',
    'profit_avg':             'Profit (average)',
    'cost_of_misestimation_avg': 'Cost misest. (avg)',
})

# Format numeric columns to 1 decimal
fmt = {col: '{:,.1f}'.format for col in df_summary.select_dtypes('float').columns}
display(df_summary.style.format(fmt).set_caption(
    f'Budget sensitivity summary (Gamma_true={GAMMA_TRUE}, In={In}, Jn={Jn}, Rn={Rn}, '
    f'V_SCALE={V_SCALE}, W={W})'
))

## 7. Plots

Three figures illustrate the trade-offs:
1. **Profit vs assumed budget** (three perspectives) — the main trade-off plot.
2. **Cost of misestimation (worst-case)** — penalty bar chart.
3. **Average profit with variability** — error-bar plot.

In [ ]:
gammas = df_summary['Gamma_assumed'].tolist()

# ---- Figure 1: profit under three scenarios vs Gamma_assumed ----
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(gammas, df_summary['Profit (no disrupt.)'],
        marker='o', linewidth=2, label='No disruption')
ax.plot(gammas, df_summary['Profit (worst-case)'],
        marker='s', linewidth=2, linestyle='--', label=f'Worst-case (\u0393_true={GAMMA_TRUE})')
ax.plot(gammas, df_summary['Profit (average)'],
        marker='^', linewidth=2, linestyle=':', label=f'Average (\u0393_true={GAMMA_TRUE})')

ax.axvline(x=GAMMA_TRUE, color='gray', linestyle='--', linewidth=1.2,
           label=f'\u0393_true = {GAMMA_TRUE} (correct design)')

ax.set_xlabel('Assumed budget \u0393_assumed', fontsize=12)
ax.set_ylabel('Profit', fontsize=12)
ax.set_title('Impact of Budget Misestimation on Profit (Section 3.4)', fontsize=13)
ax.set_xticks(gammas)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('04_budget_sensitivity_profit.pdf', bbox_inches='tight')
plt.show()

# ---- Figure 2: cost of misestimation (worst-case) ----
fig, ax = plt.subplots(figsize=(6, 4))

colors = ['#d62728' if g < GAMMA_TRUE else ('#2ca02c' if g > GAMMA_TRUE else '#1f77b4')
          for g in gammas]
bars = ax.bar([str(g) for g in gammas], df_summary['Cost misest. (WC)'],
              color=colors, edgecolor='white', linewidth=0.8)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Assumed budget \u0393_assumed', fontsize=12)
ax.set_ylabel('Cost of misestimation (worst-case)', fontsize=11)
ax.set_title('Worst-case Penalty for Wrong Budget Assumption', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Add value labels on bars
for bar, val in zip(bars, df_summary['Cost misest. (WC)']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + abs(df_summary['Cost misest. (WC)'].max()) * 0.01,
            f'{val:,.1f}', ha='center', va='bottom', fontsize=9)

# Legend patches
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#d62728', label='Underestimation'),
    Patch(facecolor='#1f77b4', label='Correct (G=2)'),
    Patch(facecolor='#2ca02c', label='Overestimation'),
]
ax.legend(handles=legend_elements, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('04_budget_sensitivity_misest_wc.pdf', bbox_inches='tight')
plt.show()

# ---- Figure 3: average profit with std error bars ----
fig, ax = plt.subplots(figsize=(6, 4))

ax.errorbar(
    gammas,
    df_avg['profit_avg'],
    yerr=df_avg['profit_std'],
    fmt='o-', linewidth=2, capsize=6, capthick=1.5,
    color='#1f77b4', ecolor='#aec7e8', label='Mean \u00b1 std'
)
ax.axvline(x=GAMMA_TRUE, color='gray', linestyle='--', linewidth=1.2,
           label=f'\u0393_true = {GAMMA_TRUE}')

ax.set_xlabel('Assumed budget \u0393_assumed', fontsize=12)
ax.set_ylabel('Average profit', fontsize=12)
ax.set_title(f'Average-case Profit \u00b1 Std Dev (N={N_SAMPLES} samples)', fontsize=12)
ax.set_xticks(gammas)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('04_budget_sensitivity_avg.pdf', bbox_inches='tight')
plt.show()

## 8. Sensitivity to congestion cost w

The congestion weight w modulates how strongly the M/M/1 waiting cost penalises overloaded facilities.
For low w, the second-stage problem is dominated by transport costs and revenue; congestion matters little.
For high w, overloaded facilities become very expensive and the value of robust hedging changes.

We repeat the worst-case analysis for w ∈ {1, 10, 100, 1000}, solving for three designs per w level:
- Nominal (G=0) — no robustness,
- Correctly-specified (G=Γ_true=2),
- Over-conservative (G=4).

The **Value of Robustness (VOR)** is defined as the improvement in worst-case profit of the correctly-specified design over the nominal design.

In [ ]:
W_VALUES         = [1.0, 10.0, 100.0, 1000.0]
GAMMA_SENS       = [0, GAMMA_TRUE, 4]   # subset for the sensitivity sweep

rows_sens = []
for w in W_VALUES:
    print(f"\n=== w = {w} ===")
    inst_w = instancemaker(In, Jn, Rn, V_SCALE, w, data_path=DATA_PATH)
    nom_w  = solve_nominal(inst_w)
    x_nom_w = nom_w['x_jr']
    print(f"  Nominal profit: {nom_w['profit']:,.1f}")

    sols_w = {}
    for G in GAMMA_SENS:
        print(f"  Solving Gamma_assumed={G}...")
        if G == 0:
            sols_w[G] = {'x_jr': x_nom_w, 'profit_design': nom_w['profit']}
        else:
            res_w = solve_CCG(
                inst_w, G, Hn,
                x_init=x_nom_w,
                tol=TOL,
                time_limit=TIME_LIMIT,
                verbose=False,
            )
            sols_w[G] = {'x_jr': res_w['x_jr'], 'profit_design': res_w['profit_LB']}
        print(f"    Design profit: {sols_w[G]['profit_design']:,.1f}")

    # Evaluate worst-case under GAMMA_TRUE for each design
    for G in GAMMA_SENS:
        print(f"  Worst-case eval for Gamma_assumed={G} (w={w})...")
        eps_wc_w, _ = worst_case_disruption(inst_w, sols_w[G]['x_jr'], GAMMA_TRUE, Hn)
        res_wc_w    = evaluate_second_stage(inst_w, sols_w[G]['x_jr'], eps_wc_w, Hn)
        rows_sens.append({
            'w':             w,
            'Gamma_assumed': G,
            'profit_design': sols_w[G]['profit_design'],
            'profit_wc':     res_wc_w['profit'],
        })
        print(f"    Worst-case profit: {res_wc_w['profit']:,.1f}")

df_sens = pd.DataFrame(rows_sens)
print("\nSensitivity sweep complete.")

In [ ]:
# Compute VOR = profit_wc(G=GAMMA_TRUE) - profit_wc(G=0) for each w
df_vor = []
for w in W_VALUES:
    sub = df_sens[df_sens['w'] == w].set_index('Gamma_assumed')
    wc_nom     = sub.loc[0,          'profit_wc']
    wc_correct = sub.loc[GAMMA_TRUE, 'profit_wc']
    wc_over    = sub.loc[4,          'profit_wc']
    vor        = wc_correct - wc_nom
    df_vor.append({
        'w':                     w,
        'Profit WC (G=0)':       wc_nom,
        f'Profit WC (G={GAMMA_TRUE})': wc_correct,
        'Profit WC (G=4)':       wc_over,
        'VOR (G=2 vs G=0)':      vor,
    })

df_vor = pd.DataFrame(df_vor)
fmt_vor = {col: '{:,.1f}'.format for col in df_vor.select_dtypes('float').columns}
display(df_vor.style.format(fmt_vor).set_caption(
    f'VOR sensitivity to congestion cost w (worst-case under Gamma_true={GAMMA_TRUE})'
))

## 9. Interpretation

### Optimal assumed budget
The correctly-specified design (\u0393_assumed = \u0393_true = 2) achieves the best **worst-case profit** by construction: C&CG optimises exactly the guarantee under budget-2 disruptions. Designs with \u0393_assumed < 2 are exposed to disruptions they were never hardened against, so the adversary finds larger profit losses. Designs with \u0393_assumed > 2 are over-hardened: they carry higher fixed costs without additional worst-case benefit relative to the threat level, and are therefore dominated under the no-disruption scenario.

### Cost of underestimation
Underestimation is typically **more costly** than overestimation in the worst-case sense. When the planner assumes \u0393 = 0 (nominal), the resulting network concentrates capacity in fewer, cheaper facilities; under a budget-2 disruption, multiple simultaneous failures can leave large customer segments unserved, dramatically reducing revenue. The cost is asymmetric: the adversary’s worst case exploits the most vulnerable open facilities.

### Cost of overestimation
Overestimation adds fixed cost without improving resilience beyond the actual threat level. The peace-time profit declines monotonically with \u0393_assumed because opening additional or more expensive facilities consumes capital that is never “necessary” at the true disruption level. However, overestimation provides a free hedge against future increases in \u0393_true (e.g., if actual disruption risk grows).

### Effect of congestion cost w
The VOR (Value of Robustness) tends to **increase with w** for moderate values because congestion amplifies the damage from demand re-routing after disruptions: when displaced customers flood surviving facilities, the queuing penalty rises sharply. At very high w the nominal design already avoids congestion by leaving slack capacity, partially mimicking a robust design, and the VOR may plateau or decline. Understanding this interaction helps practitioners calibrate the robustness level alongside their congestion-cost model.